In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

from dotenv import load_dotenv

load_dotenv()

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.max_columns = 120

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "eia930"
INTERIM_DIR = DATA_DIR / "interim" / "eia930"
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

ISO = "PJM"
START = "2023-01-01T00"
END = "2024-01-01T00"

EIA_API_KEY = os.getenv("EIA_API_KEY")

RAW_FILE = RAW_DIR / f"eia930_region_data_{ISO.lower()}_hourly_{START[:4]}.csv"
CLEAN_FILE = INTERIM_DIR / f"{ISO.lower()}_hourly_load_{START[:4]}.parquet"

RAW_FILE, CLEAN_FILE

# EIA-930 PJM Load Exploration

Purpose: build a first, reproducible view of hourly electricity demand using EIA-930 balancing authority data for one ISO/RTO. This notebook starts with **PJM** as the initial ISO because it is large, weather-sensitive, and useful for seasonal/regime-aware demand analysis.

Research guardrails:
- Use strict timestamp ordering.
- Preserve source timestamps and time zones.
- Treat downloaded data as raw input; do cleaning in explicit derived tables.
- Avoid random train/test splits.
- Always compare later models against persistence, seasonal naive, and simple linear baselines.

Primary source: EIA Open Data API, electricity/rto region data, sourced from Form EIA-930.

## Download or Load Raw EIA Data

The EIA API route used here is `electricity/rto/region-data`. The query requests `hourly` records for PJM and filters to demand (`type = D`). EIA's `hourly` frequency is UTC-oriented; `local-hourly` is the local-time alternative. If `EIA_API_KEY` is not set and no cached CSV exists, this cell will raise an instruction-oriented error rather than silently creating an incomplete dataset.

In [ ]:
def fetch_eia930_region_data(
    respondent: str,
    start: str,
    end: str,
    api_key: str | None,
    length: int = 5000,
) -> pd.DataFrame:
    """Fetch hourly EIA-930 region data for one balancing authority/ISO."""
    if not api_key:
        raise RuntimeError(
            "Set EIA_API_KEY in your environment, or place a cached raw CSV at "
            f"{RAW_FILE}."
        )

    base_url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"
    rows: list[dict] = []
    offset = 0

    while True:
        params = {
            "api_key": api_key,
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": respondent,
            "facets[type][]": "D",
            "start": start,
            "end": end,
            "sort[0][column]": "period",
            "sort[0][direction]": "asc",
            "offset": offset,
            "length": length,
        }
        response = requests.get(base_url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json().get("response", {})
        batch = payload.get("data", [])
        rows.extend(batch)

        if len(batch) < length:
            break
        offset += length

    return pd.DataFrame(rows)


if RAW_FILE.exists():
    raw = pd.read_csv(RAW_FILE)
else:
    raw = fetch_eia930_region_data(ISO, START, END, EIA_API_KEY)
    raw.to_csv(RAW_FILE, index=False)

raw.head(), raw.shape

In [ ]:
raw.info()
raw.head(10)

## Clean Load Series

This creates a minimally cleaned hourly load table. The raw file remains unchanged. We normalize timestamps to UTC and retain the original reported period for lineage.

In [ ]:
def clean_eia930_load(raw: pd.DataFrame) -> pd.DataFrame:
    required = {"period", "value"}
    missing = required.difference(raw.columns)
    if missing:
        raise ValueError(f"Missing expected EIA columns: {sorted(missing)}")

    df = raw.copy()
    df["period_raw"] = df["period"]
    df["timestamp_utc"] = pd.to_datetime(df["period"], utc=True, errors="coerce")
    df["load_mw"] = pd.to_numeric(df["value"], errors="coerce")
    df["is_missing_load"] = df["load_mw"].isna()
    df = df.dropna(subset=["timestamp_utc"])
    df = df.sort_values("timestamp_utc")
    df = df.drop_duplicates(subset=["timestamp_utc"], keep="last")

    keep = ["timestamp_utc", "period_raw", "load_mw", "is_missing_load"]
    for col in ["respondent", "respondent-name", "type", "type-name", "timezone"]:
        if col in df.columns:
            keep.append(col)
    return df[keep].reset_index(drop=True)


load = clean_eia930_load(raw)
load.to_parquet(CLEAN_FILE, index=False)
load.head(), load.tail(), load.shape

## Integrity Checks

These checks are intentionally boring. Boring checks catch expensive research mistakes.

In [ ]:
expected = pd.date_range(load["timestamp_utc"].min(), load["timestamp_utc"].max(), freq="h", tz="UTC")
observed = pd.DatetimeIndex(load["timestamp_utc"])
missing_hours = expected.difference(observed)
duplicate_hours = load["timestamp_utc"].duplicated().sum()

summary = pd.Series(
    {
        "start_utc": load["timestamp_utc"].min(),
        "end_utc": load["timestamp_utc"].max(),
        "rows": len(load),
        "missing_hours": len(missing_hours),
        "duplicate_hours": duplicate_hours,
        "null_load_rows": load["load_mw"].isna().sum(),
        "min_load_mw": load["load_mw"].min(),
        "median_load_mw": load["load_mw"].median(),
        "max_load_mw": load["load_mw"].max(),
    }
)
summary

In [ ]:
if len(missing_hours):
    display(pd.Series(missing_hours, name="missing_timestamp_utc"))

missing_load_rows = load.loc[load["is_missing_load"], ["timestamp_utc", "period_raw", "load_mw"]]
if len(missing_load_rows):
    display(missing_load_rows)


## First Visual Diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(load["timestamp_utc"], load["load_mw"], linewidth=0.8)
ax.set_title(f"{ISO} hourly demand from EIA-930")
ax.set_xlabel("UTC timestamp")
ax.set_ylabel("Load (MW)")
plt.show()

In [ ]:
load_features = load.assign(
    hour=load["timestamp_utc"].dt.hour,
    dayofweek=load["timestamp_utc"].dt.dayofweek,
    month=load["timestamp_utc"].dt.month,
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
load_features.groupby("hour")["load_mw"].mean().plot(ax=axes[0], title="Mean load by UTC hour")
load_features.groupby("dayofweek")["load_mw"].mean().plot(ax=axes[1], title="Mean load by day of week")
load_features.groupby("month")["load_mw"].mean().plot(ax=axes[2], title="Mean load by month")
for ax in axes:
    ax.set_ylabel("Load (MW)")
plt.tight_layout()
plt.show()

## Baseline-Oriented Diagnostics

Before modeling, inspect persistence and seasonal naive errors. These are not final evaluations, but they give us a baseline sanity check and reveal outages, missingness, and regime-specific failures.

In [ ]:
diagnostic_load = load.dropna(subset=["load_mw"]).copy()
diagnostics = diagnostic_load.set_index("timestamp_utc").sort_index().copy()
diagnostics["persistence_1h"] = diagnostics["load_mw"].shift(1)
diagnostics["seasonal_naive_24h"] = diagnostics["load_mw"].shift(24)
diagnostics["persistence_abs_error"] = (diagnostics["load_mw"] - diagnostics["persistence_1h"]).abs()
diagnostics["seasonal_abs_error"] = (diagnostics["load_mw"] - diagnostics["seasonal_naive_24h"]).abs()

diagnostics[["persistence_abs_error", "seasonal_abs_error"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

In [ ]:
largest_seasonal_errors = diagnostics.sort_values("seasonal_abs_error", ascending=False).head(20)
largest_seasonal_errors[["load_mw", "seasonal_naive_24h", "seasonal_abs_error"]]

In [ ]:
largest_persistence_errors = diagnostics.sort_values("persistence_abs_error", ascending=False).head(20)
largest_persistence_errors[["load_mw", "persistence_1h", "persistence_abs_error"]]

## Next Research Questions

- Are the largest seasonal naive errors concentrated in heat waves, cold snaps, holidays, or data quality events?
- Does PJM load show asymmetric response to hot versus cold temperature anomalies?
- Which horizon is the first target: 1-hour ahead, day-ahead hourly profile, or probabilistic next-day peak demand?
- What operational weather forecast data would be available at each horizon?